# testing

> A host and a backend with nothing behind either, so the harness can be driven end to end without a model.

Nothing here loads a model, and that is the bargain: what the harness is about is routing,
approval, compaction arithmetic, skill discovery, the activity stream and the tool wrappers,
and a real engine puts gigabytes and minutes in front of all of it while testing none of it.

These were leela's test fixtures. They ship now because `Host` is a thing other applications
are supposed to implement, and the clearest statement of what it requires is a host that
already satisfies it -- `MemHost` is forty lines and a dict.


In [ ]:
#| default_exp testing

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path

from ramabana.backend import Backend, Usage
from ramabana.chat import Agent
from ramabana.host import Hit, NullHost
from ramabana.models import ModelSpec

In [ ]:
#| export
#: The one spec every double reports itself as, so a status line under a fake model still
#: says something true rather than blank.
SPEC = ModelSpec('fake', 'fake', 'fake/model', ctx=1000)

In [ ]:
#| export
class MemHost(NullHost):
    "A host whose folders live in a dict, so the file tools can be driven without touching disk."

    def __init__(self, files=None, root='/proj'):
        super().__init__([root])
        self.files, self.root = dict(files or {}), root
        self.ran = []

    def check(self, path, must_exist=False):
        p = Path(path)
        return p if p.is_absolute() else Path(self.root)/p

    def walk(self): return list(self.files)
    def read(self, path): return self.files.get(str(path))
    def text_at(self, path): return self.files.get(str(path), '')

    def write(self, path, text):
        self.files[str(path)] = text
        return str(path)

    def search(self, query, limit=20):
        return [Hit(p, 1, '', t.splitlines()[0]) for p, t in self.files.items() if query in t][:limit]

    @property
    def search_note(self): return 'memory'

    def run_python(self, code):
        self.ran.append(code)
        return 'ok'

In [ ]:
#| export
class FakeBackend(Backend):
    "A backend over a scripted list of replies, so a turn can be driven with no model at all."

    kind = 'fake'

    def __init__(self, spec=SPEC, replies=(), **kw):
        super().__init__(spec, **kw)
        self.replies, self.sent, self.hist_ = list(replies), [], []
        self.spawned = []

    def _start(self): return self
    def _close(self): pass

    def _send(self, msg, **kw):
        self.sent.append(msg)
        self.hist_.append({'role': 'user', 'content': str(msg)})
        out = self.replies.pop(0) if self.replies else '(done)'
        self.hist_.append({'role': 'assistant', 'content': out})
        return out

    def _stream(self, msg, **kw):
        for w in self._send(msg, **kw).split(' '): yield w + ' '

    def _oneshot(self, prompt, sp, max_tokens): return f'ONESHOT:{prompt[:40]}'
    def _usage(self): return Usage(model=self.spec.model_id, input=10, output=5, total=15, turns=1)

    @property
    def hist(self): return self.hist_

    def _replace_hist(self, summary, keep):
        self.hist_ = [{'role': 'user', 'content': summary}] + list(keep)

    def spawn(self, sp='', tools=(), **kw):
        s = FakeBackend(self.spec, replies=['sub answer'], sp=sp, tools=tools, shared=True)
        self.spawned.append(s)
        return s

In [ ]:
#| export
def fake_agent(host=None, replies=(), **kw):
    "An `Agent` whose every job routes to one `FakeBackend`. Returns `(agent, backend)`."
    a = Agent(host or MemHost({'/proj/a.py': 'def a(): pass\n'}), extensions=False, **kw)
    be = FakeBackend(SPEC, replies=replies)
    a._be = lambda job='turn': be
    a._be_or_none = lambda job='turn': be
    return a, be

## Tests


In [ ]:
h = MemHost({'/proj/a.py': 'def a(): pass\n'})
print('walk  :', h.walk())
print('read  :', h.read('/proj/a.py').strip())
print('search:', h.search('def a'))
h.write('/proj/b.py', 'y = 2\n')
assert h.read('/proj/b.py') == 'y = 2\n'

In [ ]:
a, be = fake_agent(replies=['I looked at it.'])
print(a.ask('what is in a.py?'))
print('sent to the model:', len(be.sent), 'message(s)')
assert be.sent